# 06 -- Create Staff Addresses

Staff-migration variant of `06_Create_Addresses.ipynb` -- identical logic,
pointed at the staff pipeline's data instead. For every staff subscription
with a successfully resolved address (from
`05_Fetch_Staff_Subscriptions.ipynb`'s Voyager lookup), `PUT`s that address
onto its target account:

```
{{host}}/rest/SubscriberService/v1/subscribers/{accountcode}
```

`addLine1` / `addLine2` / `city` / `zip` / `state` (region ISO) all come from
the Voyager `ParsedAddress_*` columns -- real values, not placeholders.

Returns each new address's `id`, saved as `ship_add_id` -- this is what
`07_Create_Staff_Subscription_Orders.ipynb` uses as `shipAddId` on the order.

Subscriptions whose Voyager lookup didn't resolve are skipped here (flagged
with status `"skipped"`) -- they need a manual address before their order can
be created.

## 1. Setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_staff_addresses")

df_subscriptions = load_staff_subscriptions_resolved()
logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from 05_Fetch_Staff_Subscriptions.ipynb")


In [ ]:
# HARDCODED_TOKEN = "c5bf2481-f3fd-498b-a260-27c6e783782f"

# token_manager._token = HARDCODED_TOKEN
# token_manager._expires_at = datetime.now() + timedelta(hours=1)  # adjust to match the real token's actual TTL

## 2. Per-subscription address creation

In [ ]:
def create_address_for_subscription(session: requests.Session, row: dict) -> dict:
    subscription_id = row["SubscriptionUSN"]
    account_number  = row["TargetAccountNumber"]

    result = {
        "SubscriptionUSN":      subscription_id,
        "TargetAccountNumber":  account_number,
        "addLine1":             row.get("ParsedAddress_addLine1"),
        "status":               "failed",
        "ship_add_id":          None,
        "error":                None,
    }

    if not row.get("ParsedAddress_parsed_ok"):
        result["status"] = "skipped"
        result["error"] = row.get("ParsedAddress_error") or "Voyager address lookup failed -- needs manual address"
        return result

    status, ship_add_id, error = add_address_to_account(
        session,
        account_number,
        row["ParsedAddress_addLine1"],
        location_id=row.get("ParsedAddress_location_id") or str(subscription_id),
        address2=row.get("ParsedAddress_addLine2"),
        city=row.get("ParsedAddress_city") or "Christchurch",
        zip_code=row.get("ParsedAddress_postcode") or "1234",
        region_iso=row.get("ParsedAddress_region_iso"),
    )
    result["status"] = status
    result["ship_add_id"] = ship_add_id
    result["error"] = error

    if status == "created":
        logger.info(f"[OK] subscription {subscription_id} -> {account_number} address id={ship_add_id}")
    else:
        logger.error(f"[FAIL] subscription {subscription_id} -- {error}")

    return result


## 3. Run (parallel driver)

In [ ]:
def create_all_addresses(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    session = new_session(max_workers=max_workers)
    rows = df.to_dict("records")
    total = len(rows)
    results = []
    logger.info(f"Creating addresses for {total:,} subscriptions with {max_workers} workers...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(create_address_for_subscription, session, row): row["SubscriptionUSN"] for row in rows}
        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())
            if i % 50 == 0 or i == total:
                ok = sum(1 for r in results if r["status"] == "created")
                logger.info(f"Progress: {i}/{total} -- {ok} created so far")
    return pd.DataFrame(results)


df_address_results = create_all_addresses(df_subscriptions)
df_address_results.head(20)


## 4. Failures / skips

In [ ]:
not_created = df_address_results[df_address_results["status"] != "created"]
print(f"{len(not_created):,} / {len(df_address_results):,} addresses not created (failed or skipped)")
not_created.groupby("status").size()


## 5. Save

In [ ]:
save_df("staff_address_results", df_address_results)
